In [1]:
import pandas as pd
import numpy as np
import re
import pycountry
import matplotlib.pyplot as plt
import seaborn as sns

# import the data file
df_Data = pd.read_excel(
    r"D:\Projects\Data\Countries_Socioeconom_Profiles_Data\World_Development_Indicators.xlsx",
    sheet_name=0
)

# replace .. with nans
df_Data = df_Data.replace("..", np.nan)

# drop years with high persentage of nans
df_Data = df_Data.drop(columns=['2022 [YR2022]', '2023 [YR2023]', '2024 [YR2024]', '2025 [YR2025]'])

# Year columns
year_cols = [f"{year} [YR{year}]" for year in range(2010, 2022)]

df_Data = df_Data.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

# Extract only the year
df_Data["Year"] = df_Data["Year"].str.extract(r"(\d{4})").astype(int)

# Make values numeric
df_Data["Value"] = pd.to_numeric(df_Data["Value"], errors="coerce")

# pivot df_Data
df_wide = df_Data.pivot(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Name",
    values="Value"
).reset_index()

# normalize the names
df_wide.columns = (
    df_wide.columns
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

rename_dict = {
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Year": "year",
    
    "Access to electricity (% of population)": "access_to_electricity",
    "Agriculture, forestry, and fishing, value added (% of GDP)": "agriculture_value_added",
    "Broad money (% of GDP)": "broad_money",
    "Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)": "co2_emissions_per_capita",
    "Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)": "carbon_intensity_gdp",
    "Central government debt, total (% of GDP)": "government_debt",
    "Current health expenditure (% of GDP)": "health_expenditure",
    "Domestic credit to private sector (% of GDP)": "domestic_credit_private_sector",
    "Energy use (kg of oil equivalent per capita)": "energy_use_per_capita",
    "Exports of goods and services (% of GDP)": "exports",
    "Fertility rate, total (births per woman)": "fertility_rate",
    "Foreign direct investment, net inflows (% of GDP)": "fdi_inflows",
    "Forest area (% of land area)": "forest_area",
    "Fossil fuel energy consumption (% of total)": "fossil_fuel_consumption",
    "GDP growth (annual %)": "gdp_growth",
    "GDP per capita (constant 2015 US$)": "gdp_per_capita",
    "Gross capital formation (% of GDP)": "gross_capital_formation",
    "Imports of goods and services (% of GDP)": "imports",
    "Individuals using the Internet (% of population)": "internet_usage",
    "Industry (including construction), value added (% of GDP)": "industry_value_added",
    "Inflation, consumer prices (annual %)": "inflation",
    "Life expectancy at birth, total (years)": "life_expectancy",
    "Literacy rate, adult total (% of people ages 15 and above)": "literacy_rate",
    "Military expenditure (% of general government expenditure)": "military_expenditure",
    "Multidimensional poverty headcount ratio (World Bank) (% of population)": "multidimensional_poverty",
    "Population growth (annual %)": "population_growth",
    "Renewable electricity output (% of total electricity output)": "renewable_electricity",
    "Renewable energy consumption (% of total final energy consumption)": "renewable_energy",
    "Research and development expenditure (% of GDP)": "rd_expenditure",
    "School enrollment, primary (% gross)": "primary_school_enrollment",
    "Services, value added (% of GDP)": "services_value_added",
    "Trade (% of GDP)": "trade",
    "Unemployment, total (% of total labor force) (modeled ILO estimate)": "unemployment",
    "Urban population (% of total population)": "urban_population"
}

df_wide = df_wide.rename(columns=rename_dict)

# Retain only actual countries with valid ISO 3166-1 alpha-3 codes
valid_iso_codes = {country.alpha_3 for country in pycountry.countries}
df_countries = df_wide[df_wide['country_code'].isin(valid_iso_codes)].copy()

# calculate each country's overall percentage of missing data across all indicators and orders them from highest missingness to lowest.
feature_cols = df_countries.columns.difference(
    ["country_name", "country_code", "year"]
)

df_country_nan = (
    df_countries.groupby(["country_name", "country_code"])[feature_cols]
    .apply(lambda x: x.isna().sum().sum())
    .rename("nan_count")
    .reset_index()
)

# Total possible values per country
df_country_nan["total_values"] = (
    df_countries.groupby(["country_name", "country_code"])
    .size()
    .values * len(feature_cols)
)

# Percentage of missing values
df_country_nan["nan_percentage"] = (
    df_country_nan["nan_count"] /
    df_country_nan["total_values"]
)

df_country_nan = df_country_nan.sort_values(
    "nan_percentage",
    ascending=False
)

# delete country's with high persentage of nans
nan_per_delete = df_country_nan['nan_percentage'].quantile(q= 0.85)
country_to_delete = df_country_nan[df_country_nan['nan_percentage'] >= nan_per_delete]
country_to_delete_list = country_to_delete['country_name'].tolist()



# delete columns with high nans
nan_summary = (
    pd.DataFrame({
        "nan_count": df_countries.isna().sum(),
        "nan_percentage": df_countries.isna().mean()
    })
    .reset_index(names="Series Name")
    .sort_values("nan_percentage", ascending=False)
)

for country in country_to_delete_list:

    df_countries = df_countries[df_countries['country_name'] != country]

null_per_max = 0.5

Series_Name_delete = nan_summary[nan_summary['nan_percentage'] >= null_per_max]
Series_Name_delete_list = Series_Name_delete['Series Name'].tolist()
df_countries = df_countries.drop(columns=Series_Name_delete_list)

In [2]:
df_countries.loc[
    df_countries["fossil_fuel_energy_consumption_of_total"] < 0,
    "fossil_fuel_energy_consumption_of_total"
] = np.nan

df_countries.loc[
    df_countries["gross_capital_formation_of_gdp"] < 0,
    "gross_capital_formation_of_gdp"
] = np.nan

df_countries.drop(columns=['exports_of_goods_and_services_of_gdp', 'imports_of_goods_and_services_of_gdp'], inplace=True)

############################################################

df_countries['trade_of_gdp_log'] = np.log1p(df_countries['trade_of_gdp'])
df_countries.drop(columns=['trade_of_gdp'], inplace=True)


index_components = [
    "access_to_electricity_of_population",
    "individuals_using_the_internet_of_population",
    "life_expectancy_at_birth_total_years",
    "urban_population_of_total_population"
]

internet_col = "individuals_using_the_internet_of_population"

# 1. Country-level mean
df_countries[internet_col] = (
    df_countries[internet_col]
    .fillna(
        df_countries.groupby("country_code")[internet_col]
        .transform("mean")
    )
)

# 2. Year-level mean for countries with no Internet observations
df_countries[internet_col] = (
    df_countries[internet_col]
    .fillna(
        df_countries.groupby("year")[internet_col]
        .transform("mean")
    )
)

# 3. Global mean as absolute last fallback
df_countries[internet_col] = (
    df_countries[internet_col]
    .fillna(df_countries[internet_col].mean())
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_countries[
    [f"{col}_z" for col in index_components]
] = scaler.fit_transform(
    df_countries[index_components]
)

df_countries["Socioeconomic_developmen_Index"] = (
    df_countries[[f"{col}_z" for col in index_components]]
    .mean(axis=1)
)

df_countries.drop(columns=[
    "access_to_electricity_of_population",
    "individuals_using_the_internet_of_population",
    "life_expectancy_at_birth_total_years",
    "urban_population_of_total_population",
    'individuals_using_the_internet_of_population_z',
    'life_expectancy_at_birth_total_years_z',
    'urban_population_of_total_population_z',
    'access_to_electricity_of_population_z'
], inplace=True)

############################################################

energy_cols = [
    "energy_use_kg_of_oil_equivalent_per_capita",
    "carbon_dioxide_co2_emissions_excluding_lulucf_per_capita_t_co2ecapita"
]


# 1. Country-level mean
df_countries[energy_cols] = (
    df_countries[energy_cols]
    .fillna(
        df_countries.groupby("country_code")[energy_cols]
        .transform("mean")
    )
)

# 2. Year-level mean for countries with no Internet observations
df_countries[energy_cols] = (
    df_countries[energy_cols]
    .fillna(
        df_countries.groupby("year")[energy_cols]
        .transform("mean")
    )
)

# 3. Global mean as absolute last fallback
df_countries[energy_cols] = (
    df_countries[energy_cols]
    .fillna(df_countries[energy_cols].mean())
)


# 4. LOG TRANSFORMATION

df_log = np.log1p(df_countries[energy_cols])


# 5. STANDARDIZATION
scaler = StandardScaler()
z_cols = [f"{col}_z" for col in energy_cols]
df_countries[z_cols] = scaler.fit_transform(df_log)


# 6. CREATE ENERGY & ENVIRONMENTAL INDEX
df_countries["Energy_Environmental_Index"] = (
    df_countries[z_cols].mean(axis=1)
)


# 7. DROP ORIGINAL COMPONENTS
df_countries.drop(columns=energy_cols, inplace=True)
df_countries.drop(columns=['energy_use_kg_of_oil_equivalent_per_capita_z', 'carbon_dioxide_co2_emissions_excluding_lulucf_per_capita_t_co2ecapita_z'], inplace=True)

############################################################

df_countries['domestic_credit_to_private_sector_of_gdp_log'] = np.log1p(df_countries['domestic_credit_to_private_sector_of_gdp'])
df_countries.drop(columns= 'domestic_credit_to_private_sector_of_gdp', inplace=True)

############################################################

fdi = df_countries['foreign_direct_investment_net_inflows_of_gdp']

df_countries['fdi_log'] = np.sign(fdi) * np.log1p(np.abs(fdi))
df_countries.drop(columns='foreign_direct_investment_net_inflows_of_gdp', inplace=True)

############################################################

inflation = df_countries['inflation_consumer_prices_annual_']

df_countries['inflation_log'] = (np.sign(inflation)* np.log1p(np.abs(inflation)))
df_countries.drop(columns='inflation_consumer_prices_annual_', inplace=True)

############################################################

df_countries['broad_money_of_gdp_log'] = np.log1p(df_countries['broad_money_of_gdp'])
df_countries.drop(columns='broad_money_of_gdp', inplace=True)

############################################################

df_countries['gdp_per_capita_constant_2015_us_log'] = np.log1p(df_countries['gdp_per_capita_constant_2015_us'])
df_countries.drop(columns='gdp_per_capita_constant_2015_us', inplace=True)

############################################################

df_countries['military_expenditure_of_general_government_expenditure_log'] = np.log1p(df_countries['military_expenditure_of_general_government_expenditure'])
df_countries.drop(columns='military_expenditure_of_general_government_expenditure', inplace=True)

In [3]:
low_missing_cols = [
    'agriculture_forestry_and_fishing_value_added_of_gdp',
    'carbon_intensity_of_gdp_kg_co2e_per_constant_2015_us_of_gdp',
    'current_health_expenditure_of_gdp',
    'forest_area_of_land_area',
    'gdp_growth_annual_',
    'industry_including_construction_value_added_of_gdp',
    'renewable_electricity_output_of_total_electricity_output',
    'school_enrollment_primary_gross',
    'services_value_added_of_gdp',
    'unemployment_total_of_total_labor_force_modeled_ilo_estimate',
    'domestic_credit_to_private_sector_of_gdp_log',
    'fdi_log',
    'inflation_log',
    'gdp_per_capita_constant_2015_us_log'
]

df_countries = (
    df_countries
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)


for col in low_missing_cols:

    # ==================================================
    # 1. Interpolate small internal gaps within country
    # ==================================================

    df_countries[col] = (
        df_countries
        .groupby("country_code")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit=3,
                limit_area="inside"
            )
        )
    )

    # ==================================================
    # 2. Calculate each country's typical value
    # ==================================================

    country_medians = (
        df_countries
        .groupby("country_code")[col]
        .median()
    )

    # ==================================================
    # 3. Assign countries to deciles
    # ==================================================

    country_groups = pd.qcut(
        country_medians.rank(method="first"),
        q=10,
        labels=[f"Decile_{i}" for i in range(1, 11)]
    )

    # ==================================================
    # 4. Calculate median of country medians
    #    within each decile
    # ==================================================

    decile_medians = (
        country_medians
        .groupby(country_groups, observed=True)
        .median()
    )

    # ==================================================
    # 5. Map each country's decile to its rows
    # ==================================================

    row_groups = df_countries["country_code"].map(country_groups)

    # ==================================================
    # 6. Impute using country's decile median
    # ==================================================

    decile_imputation = row_groups.map(decile_medians)

    df_countries[col] = (
        df_countries[col]
        .fillna(decile_imputation)
    )

    # ==================================================
    # 7. If country has no historical data → year median
    # ==================================================

    df_countries[col] = (
        df_countries[col]
        .fillna(
            df_countries
            .groupby("year")[col]
            .transform("median")
        )
    )

    # ==================================================
    # 8. Absolute final fallback → global median
    # ==================================================

    df_countries[col] = (
        df_countries[col]
        .fillna(df_countries[col].median())
    )

############################################################

# ==============================================================================
# HIGH-MISSING INDICATORS
# ==============================================================================

high_missing_cols = [
    "fossil_fuel_energy_consumption_of_total",
    "military_expenditure_of_general_government_expenditure_log",
    "broad_money_of_gdp_log",
    "gross_capital_formation_of_gdp",
    "trade_of_gdp_log",
]

anchor_col = "gdp_per_capita_constant_2015_us_log"


# ==============================================================================
# 1. CREATE IMPUTATION METHOD COLUMNS
# ==============================================================================

imputation_method_cols = {}

for col in high_missing_cols:

    method_col = f"{col}_imputation_method"

    imputation_method_cols[col] = method_col

    df_countries[method_col] = np.where(
        df_countries[col].isna(),
        "missing",
        "observed"
    )


# ==============================================================================
# 2. STRUCTURAL ZERO
# ==============================================================================

no_military_countries = [
    "CRI",
    "ISL",
    "KIR",
    "FSM",
    "MUS",
    "PAN"
]

military_col = "military_expenditure_of_general_government_expenditure_log"

mask_structural_zero = (
    df_countries["country_code"].isin(no_military_countries)
    & df_countries[military_col].isna()
)

df_countries.loc[
    mask_structural_zero,
    military_col
] = 0.0

df_countries.loc[
    mask_structural_zero,
    imputation_method_cols[military_col]
] = "structural_zero"


# ==============================================================================
# 3. PREPARE GDP-PER-CAPITA ANCHOR
# ==============================================================================

clean_anchor_series = (
    df_countries
    .groupby("country_code")[anchor_col]
    .transform(
        lambda x: x.interpolate(
            method="linear",
            limit=3,
            limit_area="inside"
        )
    )
)

country_anchor_medians = (
    clean_anchor_series
    .groupby(df_countries["country_code"])
    .transform("median")
)

global_anchor_median = clean_anchor_series.median()

clean_anchor_series = (
    clean_anchor_series
    .fillna(country_anchor_medians)
    .fillna(global_anchor_median)
)

country_anchor_ranks = (
    clean_anchor_series
    .groupby(df_countries["country_code"])
    .median()
)


# ==============================================================================
# 4. IMPUTE HIGH-MISSING INDICATORS
# ==============================================================================

for col in high_missing_cols:

    method_col = imputation_method_cols[col]

    # --------------------------------------------------------------------------
    # Save ORIGINAL fallback statistics
    # --------------------------------------------------------------------------

    raw_global_median = df_countries[col].median()

    raw_year_medians = (
        df_countries
        .groupby("year")[col]
        .transform("median")
    )

    # --------------------------------------------------------------------------
    # STEP A — LINEAR INTERPOLATION
    # --------------------------------------------------------------------------

    before = df_countries[col].isna()

    df_countries[col] = (
        df_countries
        .groupby("country_code")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit=3,
                limit_area="inside"
            )
        )
    )

    mask = before & df_countries[col].notna()

    df_countries.loc[
        mask,
        method_col
    ] = "interpolation"


    # --------------------------------------------------------------------------
    # STEP B — COUNTRY MEDIANS
    # --------------------------------------------------------------------------

    country_medians = (
        df_countries
        .groupby("country_code")[col]
        .median()
    )


    # --------------------------------------------------------------------------
    # STEP C — COUNTRY RANKING
    #
    # If the country has data for this indicator:
    #     rank using its own median
    #
    # If the country has NO data:
    #     rank using GDP-per-capita anchor
    # --------------------------------------------------------------------------

    ranking_series = (
        country_medians
        .fillna(country_anchor_ranks)
    )


    # --------------------------------------------------------------------------
    # STEP D — CREATE DECILES
    # --------------------------------------------------------------------------

    valid_ranking = ranking_series.dropna()

    n_quantiles = min(
        10,
        len(valid_ranking)
    )

    if n_quantiles >= 2:

        country_groups = pd.qcut(
            ranking_series.rank(method="first"),
            q=n_quantiles,
            labels=[
                f"Decile_{i}"
                for i in range(1, n_quantiles + 1)
            ],
            duplicates="drop"
        )

    else:

        country_groups = pd.Series(
            index=ranking_series.index,
            dtype="object"
        )


    # --------------------------------------------------------------------------
    # STEP E — MAP COUNTRY → DECILE
    # --------------------------------------------------------------------------

    row_groups = df_countries["country_code"].map(
        country_groups
    )


    # --------------------------------------------------------------------------
    # STEP F — CALCULATE DECILE MEDIANS
    # --------------------------------------------------------------------------

    decile_medians = (
        df_countries
        .groupby(
            row_groups,
            observed=True
        )[col]
        .median()
    )

    decile_imputation = row_groups.map(
        decile_medians
    )


    # --------------------------------------------------------------------------
    # STEP G — DECILE MEDIAN IMPUTATION
    # --------------------------------------------------------------------------

    before = df_countries[col].isna()

    df_countries[col] = (
        df_countries[col]
        .fillna(decile_imputation)
    )

    mask = before & df_countries[col].notna()

    df_countries.loc[
        mask,
        method_col
    ] = "decile_median"


    # --------------------------------------------------------------------------
    # STEP H — YEARLY CROSS-SECTIONAL MEDIAN
    # --------------------------------------------------------------------------

    before = df_countries[col].isna()

    df_countries[col] = (
        df_countries[col]
        .fillna(raw_year_medians)
    )

    mask = before & df_countries[col].notna()

    df_countries.loc[
        mask,
        method_col
    ] = "yearly_median"


    # --------------------------------------------------------------------------
    # STEP I — GLOBAL MEDIAN
    # --------------------------------------------------------------------------

    before = df_countries[col].isna()

    df_countries[col] = (
        df_countries[col]
        .fillna(raw_global_median)
    )

    mask = before & df_countries[col].notna()

    df_countries.loc[
        mask,
        method_col
    ] = "global_median"

### **Now ready to start the model**